# Pipeline Demo — A1 Baseline

Short walkthrough of the EchoBot and EK80 signal-processing pipelines on a single A1 baseline run. This notebook is intentionally minimal: load one EchoBot `.mat`, load one EK80 `.raw`, process both, and overlay the results. Runtime is under two minutes on a laptop.

For the full nine-group reproduction of every manuscript figure, see [`02_manuscript_analysis.ipynb`](02_manuscript_analysis.ipynb).

## Requirements
- A1 baseline EchoBot `.mat` at `../data/echobot/0311-CRL-tests/backcyl_bis_rgh0.01271_T115300_100.mat`
- A1 baseline EK80 `.raw` at `../data/EK80/0311-CRL-tests/prod-D20260311-T182413.raw`
- `NOAA-381-WC-TSf.xlsx` in `../data/` (already bundled)

Both raw files are in the Zenodo release (see [`docs/DATA.md`](../docs/DATA.md)).

In [ ]:
%matplotlib inline
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path

from echobot import io, echobot_pipeline, ek80_pipeline, compare, plotting, config

DATA = Path('..') / 'data'
EB_FILE = DATA / 'echobot' / '0311-CRL-tests' / 'backcyl_bis_rgh0.01271_T115300_100.mat'
EK_FILE = DATA / 'EK80' / '0311-CRL-tests' / 'prod-D20260311-T182413.raw'
NOAA_FILE = DATA / 'NOAA-381-WC-TSf.xlsx'

for p in (EB_FILE, EK_FILE, NOAA_FILE):
    print(f'{"OK " if p.exists() else "MISSING ":9s}{p}')

## 1. EchoBot pipeline — uncalibrated TS(f)

`echobot.echobot_pipeline.process_echobot_run` runs steps 1–6 of the manuscript Section 2.4 pipeline end-to-end:

1. Load `.mat` → header + sample array
2. Build zero-padded transmit reference
3. Bandpass (FIR LP 175 kHz + HP 80 kHz)
4. Matched filter (3-sector sum)
5. Range gate around autodetected target peak (±0.40 m)
6. Coherent-averaged FFT + `|TX|²` deconvolution → uncalibrated TS(f)

In [ ]:
eb_run = io.load_echobot_mat(EB_FILE)
print(f'EB run: fs={eb_run.fs/1e3:.0f} kHz, n_pings={eb_run.n_pings}, n_samples={eb_run.n_samples}')

eb_proc = echobot_pipeline.process_echobot_run(eb_run)
print(f'  target_center = {eb_proc.target_center_m:.4f} m')
print(f'  floor_center  = {eb_proc.floor_center_m:.4f} m')
print(f'  TS(f) length  = {len(eb_proc.tsf_db)} (NFFT)')

mid_mask = (eb_proc.f_hz >= 100e3) & (eb_proc.f_hz <= 140e3)
print(f'  EB uncalibrated mean TS (100-140 kHz) = {eb_proc.tsf_db[mid_mask].mean():+.2f} dB')

In [ ]:
# Quick look: matched-filter envelope for ping 0 and TS(f) spectrum
from scipy.signal import hilbert

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(13, 4))

env0 = np.abs(hilbert(eb_proc.all_mf[0]))
env0_db = 20 * np.log10(env0 / env0.max() + 1e-30)
ax1.plot(eb_proc.r_mf, env0_db, color=plotting.EB_COLOR, lw=1.0)
ax1.axvspan(eb_proc.target_center_m - config.GATE_HALF_M,
            eb_proc.target_center_m + config.GATE_HALF_M,
            alpha=0.15, color=plotting.EB_COLOR, label='target gate')
ax1.set_xlim(0.5, 3.5)
ax1.set_ylim(-60, 2)
ax1.set_xlabel('Range (m)')
ax1.set_ylabel('Normalized envelope (dB)')
ax1.set_title('EB ping 0 matched-filter envelope')
ax1.grid(True, alpha=0.3)
ax1.legend(fontsize=9)

band = (eb_proc.f_hz >= 90e3) & (eb_proc.f_hz <= 150e3)
ax2.plot(eb_proc.f_hz[band]/1e3, eb_proc.tsf_db[band], color=plotting.EB_COLOR, lw=1.5)
ax2.set_xlabel('Frequency (kHz)')
ax2.set_ylabel('TS(f) (dB, uncalibrated)')
ax2.set_title('EB coherent-avg TS(f)')
ax2.grid(True, alpha=0.3)

fig.tight_layout()

## 2. EK80 pipeline — calibrated absolute TS(f)

`echobot.ek80_pipeline.process_ek80_ed` runs steps 1–8 of Section 2.5:

1. Load `.raw` via echopype
2. Reconstruct RF transmit chirp with Hanning edge taper
3. Apply WBT + PC filter cascade (from `.raw` vendor metadata) → `tx_filt`, `norm_fac`
4. Pulse-compress every ping (mean across transducer beams)
5. Range gate, Hanning, coherent-average FFT
6. Aliasing-aware frequency mapping (baseband `fs` < chirp bandwidth)
7. `|TX|²` deconvolution
8. Full sonar equation via `compute_absolute_tsf`

The full sonar equation and the parameter sources are documented in [`docs/SONAR_EQUATION.md`](../docs/SONAR_EQUATION.md).

In [ ]:
ed = io.open_ek80_raw(EK_FILE)
ek_proc = ek80_pipeline.process_ek80_ed(ed)
print(f'EK80 run: fs_bb={ek_proc.meta.fs_bb_hz/1e3:.2f} kHz, n_pings={ek_proc.meta.n_pings}')
print(f'  chirp {ek_proc.meta.f_start_hz/1e3:.0f}-{ek_proc.meta.f_stop_hz/1e3:.0f} kHz, '
      f'dur={ek_proc.meta.t_dur_s*1e3:.2f} ms')
print(f'  Ptx={ek_proc.meta.transmit_power_w} W, G={ek_proc.meta.gain_correction_db} dB')
print(f'  target_center = {ek_proc.target_center_m:.4f} m')
print(f'  norm_fac = {ek_proc.norm_fac:.3f}  (20·log10 = {20*np.log10(ek_proc.norm_fac):+.2f} dB)')

ek_mid = (ek_proc.f_band_sorted >= 100e3) & (ek_proc.f_band_sorted <= 140e3)
ts_120 = float(np.interp(120e3, ek_proc.f_band_sorted, ek_proc.tsf_calibrated_db))
print(f'  EK calibrated mean TS (100-140 kHz) = {ek_proc.tsf_calibrated_db[ek_mid].mean():+.2f} dB')
print(f'  EK calibrated TS @ 120 kHz         = {ts_120:+.2f} dB  (expected: -37.14)')

## 3. Cross-instrument comparison

Both pipelines produce a TS(f) spectrum over 90–150 kHz. The manuscript's main-text comparison uses mean-subtracted (normalized) spectra on the 100–140 kHz overlap band to isolate spectral shape agreement. The Pearson correlation here should match the manuscript Table 3 A1 value of ≈0.977.

In [ ]:
res = compare.cross_instrument_r(
    eb_proc.tsf_db, eb_proc.f_hz,
    ek_proc.tsf_calibrated_db, ek_proc.f_band_sorted,
    f_lo_hz=100e3, f_hi_hz=140e3,
)
print(f'Cross-instrument Pearson r (100-140 kHz): {res.pearson_r:.4f}')

fig = plotting.plot_baseline_comparison(eb_proc, ek_proc, title='A1 Baseline Cross-Instrument Comparison')
plt.show()

## 4. Absolute TS(f) overlay vs NOAA theory

A third reference curve is the NOAA theoretical TS(f) for the 38.1 mm tungsten carbide calibration sphere (bundled in `data/NOAA-381-WC-TSf.xlsx`). Overlaying the EchoBot uncalibrated spectrum, the EK80 calibrated spectrum, and the NOAA theoretical spectrum on one absolute dB axis shows:
- The ~+37 dB offset between uncalibrated EchoBot and calibrated EK80 (the unknown EchoBot electronics gain constant)
- The ~+3 dB offset between calibrated EK80 and theory (residual spectral tilt from the scalar-gain approximation)

In [ ]:
noaa = pd.read_excel(NOAA_FILE)
noaa.columns = [c.strip() for c in noaa.columns]
f_noaa = noaa['Frequency'].values * 1e3
ts_noaa = noaa['dB (*-1)'].values

fig = plotting.plot_absolute_tsf(
    eb_proc.f_hz, eb_proc.tsf_db,
    ek_proc.f_band_sorted, ek_proc.tsf_calibrated_db,
    f_noaa, ts_noaa,
)
plt.show()

## Summary

With a ~2-minute run, we've:
- Reproduced the A1 baseline EchoBot uncalibrated TS(f) (mean ≈ −0.6 dB over 100–140 kHz)
- Reproduced the A1 baseline EK80 calibrated TS(f) (TS@120 kHz ≈ −37.14 dB, matching the EK80 desktop software to within 0.54 dB)
- Computed the cross-instrument Pearson r ≈ 0.98 over the 100–140 kHz overlap band
- Recovered the ~+37 dB uncalibrated-EB ↔ calibrated-EK offset and the ~+3 dB EK ↔ NOAA-theory offset

For the full nine-group reproduction (A1, A1-dup, A1-rep, B1, C1, D1, P1-lo/mid/hi), SNR analyses, linearity diagnostics, and all supplementary figures, proceed to [`02_manuscript_analysis.ipynb`](02_manuscript_analysis.ipynb).